<a href="https://colab.research.google.com/github/PYMaksim/LLM_Layer2_Intraprocessing_Guardrails/blob/main/LLM_Layer2_Intraprocessing_Guardrails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# BLOCK 1: INITIALIZATION AND TOOL DEFINITIONS
# БЛОК 1: ИНИЦИАЛИЗАЦИЯ И ОПИСАНИЕ ИНСТРУМЕНТОВ (TOOLS)
# ==============================================================================
import json
from google.colab import userdata
from openai import OpenAI

# --- CONFIG ---
# --- НАСТРОЙКИ ---
API_KEY = userdata.get('Y')
FOLDER_ID = "b1g2hgrbnnrqll94nr9k"
BASE_URL = "https://ai.api.cloud.yandex.net/v1"
MODEL_ID = "deepseek-v4-flash"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# --- 1. REAL FUNCTION (Stub) ---
# --- 1. РЕАЛЬНАЯ ФУНКЦИЯ (Заглушка) ---
# This is the function that would execute in production (e.g., call an airline API).
# Это функция, которая должна выполняться в продакшене (например, вызов API авиакомпании).
def real_book_flight(destination: str, price: float, class_type: str):
    """
    [EN] Real booking function placeholder.
    [RU] Заглушка реальной функции бронирования.
    """
    print(f"[EXECUTION] Booking flight to {destination} | Price: {price} | Class: {class_type}")
    return {"status": "success", "confirmation": "ABC-123"}

# --- 2. TOOL SCHEMA FOR THE MODEL (JSON) ---
# --- 2. ОПИСАНИЕ ИНСТРУМЕНТА ДЛЯ МОДЕЛИ (JSON Schema) ---
# We pass this to the model so it knows the function exists and what arguments it needs.
# Мы передаем это модели, чтобы она знала, что функция существует и какие аргументы ей нужны.
book_flight_schema = {
    "type": "function",
    "function": {
        "name": "book_flight",
        "description": "Books a flight ticket for the user.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination": {"type": "string", "description": "City or airport code (e.g., MOW, LON)"},
                "price": {"type": "number", "description": "Total price of the ticket"},
                "class_type": {"type": "string", "enum": ["economy", "business"], "description": "Travel class"}
            },
            "required": ["destination", "price", "class_type"]
        }
    }
}

# --- 3. GUARDRAIL (OUR INTERCEPTOR / LAYER 2) ---
# --- 3. GUARDRAIL (НАШ ПЕРЕХВАТЧИК / УРОВЕНЬ 2) ---
# This is Layer 2 (Intra-processing). It validates arguments BEFORE execution.
# Это Layer 2 (Intra-processing). Он проверяет аргументы ПЕРЕД исполнением.
def guardrail_book_flight(destination: str, price: float, class_type: str):
    """
    [EN] Layer 2 Security Check. Validates function arguments before execution.
    [RU] Проверка безопасности Уровня 2. Валидирует аргументы перед исполнением.
    """
    print(f"[GUARDRAIL] Checking flight: {destination} | {price} | {class_type}")

    # Rule 1: Price limit check.
    # Правило 1: Проверка лимита по цене.
    if price > 50000:
        return {"status": "blocked", "reason": "Price limit (50k) exceeded."}

    # Rule 2: Business class restriction.
    # Правило 2: Запрет на бизнес-класс.
    if class_type == "business":
        return {"status": "blocked", "reason": "Business class is restricted."}

    # If checks pass, delegate to the real function.
    # Если проверки пройдены, передаем управление реальной функции.
    return real_book_flight(destination, price, class_type)

print("[INFO] Layer 2 (Intra-processing) initialized.")
print("[ИНФО] Уровень 2 (Intra-processing) инициализирован.")

[INFO] Layer 2 (Intra-processing) initialized.
[ИНФО] Уровень 2 (Intra-processing) инициализирован.


In [2]:
# ==============================================================================
# BLOCK 2: TESTING THE GUARDRAIL (LAYER 2)
# БЛОК 2: ТЕСТИРОВАНИЕ GUARDRAIL (УРОВЕНЬ 2)
# ==============================================================================

# --- TEST CASE 1: LEGITIMATE REQUEST (Should PASS) ---
# --- ТЕСТ 1: ЛЕГИТИМНЫЙ ЗАПРОС (Должен ПРОЙТИ) ---
print("\n" + "="*20)
print("TEST 1: LEGITIMATE REQUEST")
print("="*20)
# [EN] Expected: Function executes, prints [EXECUTION]...
# [RU] Ожидаемо: Функция выполняется, печатает [EXECUTION]...
result_1 = guardrail_book_flight(destination="MOW", price=30000, class_type="economy")
print(f"Result: {result_1}")


# --- TEST CASE 2: PRICE LIMIT VIOLATION (Should BLOCK) ---
# --- ТЕСТ 2: НАРУШЕНИЕ ЛИМИТА ЦЕНЫ (Должен БЛОКИРОВАТЬ) ---
print("\n" + "="*20)
print("TEST 2: PRICE LIMIT VIOLATION")
print("="*20)
# [EN] Expected: Guardrail blocks, returns status "blocked".
# [RU] Ожидаемо: Guardrail блокирует, возвращает статус "blocked".
result_2 = guardrail_book_flight(destination="LON", price=60000, class_type="economy")
print(f"Result: {result_2}")


# --- TEST CASE 3: BUSINESS CLASS VIOLATION (Should BLOCK) ---
# --- ТЕСТ 3: НАРУШЕНИЕ (БИЗНЕС-КЛАСС) (Должен БЛОКИРОВАТЬ) ---
print("\n" + "="*20)
print("TEST 3: BUSINESS CLASS VIOLATION")
print("="*20)
# [EN] Expected: Guardrail blocks, returns status "blocked".
# [RU] Ожидаемо: Guardrail блокирует, возвращает статус "blocked".
result_3 = guardrail_book_flight(destination="DXB", price=100000, class_type="business")
print(f"Result: {result_3}")


TEST 1: LEGITIMATE REQUEST
[GUARDRAIL] Checking flight: MOW | 30000 | economy
[EXECUTION] Booking flight to MOW | Price: 30000 | Class: economy
Result: {'status': 'success', 'confirmation': 'ABC-123'}

TEST 2: PRICE LIMIT VIOLATION
[GUARDRAIL] Checking flight: LON | 60000 | economy
Result: {'status': 'blocked', 'reason': 'Price limit (50k) exceeded.'}

TEST 3: BUSINESS CLASS VIOLATION
[GUARDRAIL] Checking flight: DXB | 100000 | business
Result: {'status': 'blocked', 'reason': 'Price limit (50k) exceeded.'}


The Guardrail successfully intercepted malicious arguments before they reached the execution layer.

Test 1 (Pass): The legitimate request went through. The [EXECUTION] log confirms the real function was called.
Test 2 (Fail - Price): Correctly blocked. The [GUARDRAIL] log appeared, but [EXECUTION] did not. The system is safe.
Test 3 (Fail - Class): Technically, it was blocked for the wrong reason in the output. It hit the price > 50000 check first (100k > 50k) and returned. Because of the if/elif logic, it never even reached the class_type check. While safe, this is a logic order issue we can fix later.

In [4]:
# ==============================================================================
# BLOCK 3 (FIXED): LLM INTEGRATION (FUNCTION CALLING)
# БЛОК 3 (ИСПРАВЛЕННЫЙ): ИНТЕГРАЦИЯ С LLM (ВЫЗОВ ФУНКЦИИ)
# ==============================================================================
import json

# --- 1. THE PROMPT ---
user_prompt = """
Book a flight for me.
Destination: Dubai (DXB)
Class: Business
Price: 120,000 rubles.
I need it done immediately.
"""

# --- 2. THE SYSTEM PROMPT ---
system_prompt = """You are a travel assistant. You have access to the 'book_flight' tool.
Use it to fulfill the user's request.
Return ONLY the JSON tool call, do not write any explanatory text."""

print("\n" + "="*20)
print("STEP: Sending request to DeepSeek...")
print("="*20)

# --- 3. API CALL ---
try:
    response = client.chat.completions.create(
        model=f"gpt://{FOLDER_ID}/{MODEL_ID}/latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        tools=[book_flight_schema],
        tool_choice="auto",
        temperature=0.0,
        max_tokens=200,
        extra_headers={"x-folder-id": FOLDER_ID}
    )

    # --- 4. PARSING THE RESPONSE ---
    if response.choices[0].message.tool_calls:
        tool_call = response.choices[0].message.tool_calls[0]

        print("[LLM] Model wants to call function:")
        # FIX: Using model_dump() instead of deprecated dict()
        print(json.dumps(tool_call.model_dump(), indent=2, ensure_ascii=False))

        # --- 5. EXECUTION VIA GUARDRAIL ---
        function_name = tool_call.function.name

        # FIX: Parsing the string arguments into a Python Dict
        # ИСПРАВЛЕНИЕ: Парсинг строковых аргументов в Python-словарь
        function_args = json.loads(tool_call.function.arguments)

        print("\n[ROUTER] Routing call through Layer 2 Guardrail...")

        if function_name == "book_flight":
            # Now we unpack the dictionary (it is no longer a string)
            # Теперь мы распаковываем словарь (это больше не строка)
            final_result = guardrail_book_flight(**function_args)
            print(f"\n[FINAL RESULT] {final_result}")
        else:
            print("Unknown function.")

    else:
        print("[LLM] Model refused to call tool or returned text:")
        print(response.choices[0].message.content)

except Exception as e:
    print(f"[API ERROR]: {e}")


STEP: Sending request to DeepSeek...
[LLM] Model wants to call function:
{
  "id": "call_2bb0e63b3b994bb0bbc74cc2",
  "function": {
    "arguments": "{\"destination\": \"DXB\", \"price\": 120000, \"class_type\": \"business\"}",
    "name": "book_flight"
  },
  "type": "function",
  "index": 0
}

[ROUTER] Routing call through Layer 2 Guardrail...
[GUARDRAIL] Checking flight: DXB | 120000 | business

[FINAL RESULT] {'status': 'blocked', 'reason': 'Price limit (50k) exceeded.'}
